### The GIL — Start Here, Everything Follows From It

**Global Interpreter Lock** — one lock that protects Python's memory manager.

Only **ONE thread** can execute Python bytecode **at a time**.

**Not one per core — one total, globally, across all threads**.
```shell
Thread 1 ──→ acquires GIL ──→ runs ──→ releases GIL
Thread 2 ──→ waiting ─────────────────→ acquires GIL ──→ runs
Thread 3 ──→ waiting ──────────────────────────────────→ ...
```
Consequence:
  - CPU-bound work  + threading  = **NO** speedup    (GIL blocks parallelism)
  - I/O-bound work  + threading  = **YES** speedup   (GIL released during I/O wait)
  - CPU-bound work  + multiprocessing = **YES** speedup (separate processes, no GIL)

---

### Concurrency vs Parallelism — The Distinction
```shell
CONCURRENCY  — dealing with multiple things at once (structure)
               tasks interleave, one CPU, cooperative or preemptive
               asyncio, threading

PARALLELISM  — doing multiple things at once (execution)
               multiple CPUs running simultaneously
               multiprocessing, C extensions that release GIL

asyncio ──→ concurrency, single thread
threading ─→ concurrency, multiple threads, one GIL
multiprocessing → parallelism, multiple processes, no GIL
```

---

**`asyncio` Mental Model — The Event Loop**

**The event loop is a SINGLE thread running a loop**

In [ ]:
import asyncio

# The event loop is a SINGLE thread running a loop:
#
# while True:
#     event = get_next_ready_event()
#     run_callback(event)
#
# It never blocks. When a coroutine hits `await`, it SUSPENDS itself
# and hands control back to the loop → loop runs something else.
# When the awaited thing finishes, loop RESUMES the coroutine.

#                  ┌─────────────────────────────┐
#                  │         Event Loop          │
#                  │                             │
#  coroutine A ────┤ await sleep(1) ────────┐    │
#                  │                        │    │
#  coroutine B ────┤ await db.fetch() ──┐   │    │
#                  │                    │   │    │
#  coroutine C ────┤ runs...            │   │    │
#                  │                    │   │    │
#                  │ I/O ready ─────────┘   │    │  ← B resumes
#                  │ timer done ────────────┘    │  ← A resumes
#                  └─────────────────────────────┘
#
# Key: COOPERATIVE — coroutines yield control willingly at await points
#      No preemption — a coroutine that never awaits blocks everything

**Key Objects — Coroutine, Task, Future**

In [6]:
import asyncio

# ── 1. COROUTINE ──────────────────────────────────────────────
# A function defined with async def
# Calling it returns a coroutine OBJECT — nothing runs yet

async def fetch_data(url: str) -> dict:
    await asyncio.sleep(1)    # ← suspension point
    return {"url": url, "data": "..."}

In [7]:
coro = fetch_data("http://example.com")
print(type(coro))   # <class 'coroutine'>
print(coro)         # <coroutine object fetch_data at 0x...>
# fetch_data body has NOT run yet — it's just an object

<class 'coroutine'>
<coroutine object fetch_data at 0x0000016C36152F80>


C:\Users\Dennie\AppData\Local\Temp\ipykernel_7972\1974168072.py:1: RuntimeWarning: coroutine 'fetch_data' was never awaited
  coro = fetch_data("http://example.com")


In [13]:
# ── 2. TASK ───────────────────────────────────────────────────
# Wraps a coroutine and SCHEDULES it on the event loop immediately
# The loop will run it as soon as possible

async def main():
    task = asyncio.create_task(fetch_data("http://example.com"))
    # ↑ fetch_data is NOW scheduled — will run concurrently
    # task is a Future-like object you can await, cancel, check

    print(task.done())     # False — still running
    result = await task    # suspend main until task completes
    print(task.done())     # True
    print(result)          # {"url": "...", "data": "..."}

In [ ]:
await main()  # don't use asyncio.run(). Jupyter uses IPython, and IPython starts an event loop automatically

False
True
{'url': 'http://example.com', 'data': '...'}


In [ ]:
# ── 3. FUTURE ─────────────────────────────────────────────────
# Low-level object representing a value not yet computed
# Task is a subclass of Future
# You rarely create Futures manually — asyncio internals use them

async def low_level_example():
    loop   = asyncio.get_event_loop()
    future = loop.create_future()          # empty future

    loop.call_soon(future.set_result, 42)  # schedule: set result to 42
    result = await future                  # suspend until result is set
    print(result)                          # 42


#  Coroutine → the async function call (object, not yet running)
#  Task      → coroutine wrapped + scheduled on the loop (running)
#  Future    → raw promise of a value (Task inherits from this)

**`await` — What It Actually Does**

In [ ]:
async def main():
    result = await some_coroutine()
    #        ↑
    #  1. Suspends main() — hands control BACK to event loop
    #  2. Event loop runs other tasks while waiting
    #  3. When some_coroutine() finishes → loop resumes main()
    #  4. result = the return value of some_coroutine()


# ── What you can await: ───────────────────────────────────────
await coroutine_function()   # any async def call
await asyncio.sleep(1)       # async sleep (never block with time.sleep!)
await task                   # wait for a scheduled task
await future                 # wait for a future's result
await asyncio.gather(...)    # wait for multiple things


# ── The cardinal sin: ─────────────────────────────────────────
import time

async def broken():
    time.sleep(2)       # ❌ BLOCKS the entire event loop
                        #    no other coroutine can run for 2 seconds
    await asyncio.sleep(2)  # ✅ suspends THIS coroutine, others run freely

**`asyncio.gather` — Run Concurrently, Collect Results**

In [10]:
import asyncio
import time


async def fetch_user(user_id: int) -> dict:
    print(f"  fetching user {user_id}...")
    await asyncio.sleep(1)          # simulate network I/O
    return {"id": user_id, "name": f"User {user_id}"}

async def fetch_orders(user_id: int) -> list:
    print(f"  fetching orders for {user_id}...")
    await asyncio.sleep(1.5)        # simulate slower query
    return [{"order_id": 1}, {"order_id": 2}]


# ── Sequential — slow ─────────────────────────────────────────
async def sequential():
    start = time.perf_counter()
    user   = await fetch_user(1)     # wait 1s
    orders = await fetch_orders(1)   # wait 1.5s after first finishes
    print(f"Sequential: {time.perf_counter() - start:.2f}s")
    # → ~2.5s total


# ── Concurrent with gather — fast ─────────────────────────────
async def concurrent():
    start = time.perf_counter()
    user, orders = await asyncio.gather(
        fetch_user(1),       # starts immediately
        fetch_orders(1),     # starts immediately, runs alongside
    )
    print(f"Concurrent: {time.perf_counter() - start:.2f}s")
    # → ~1.5s total (limited by slowest, not sum)

In [ ]:
await sequential()

  fetching user 1...
  fetching orders for 1...
Sequential: 2.52s


In [16]:
await concurrent()

  fetching user 1...
  fetching orders for 1...
Concurrent: 1.50s


In [21]:
# ── gather with error handling ────────────────────────────────
async def risky_fetch(n: int) -> int:
    if n == 2:
        raise ValueError(f"Failed on {n}")
    return n * 10

async def handle_errors():
    # return_exceptions=True → exceptions returned as values, not raised
    results = await asyncio.gather(
        risky_fetch(1),
        risky_fetch(2),     # this fails
        risky_fetch(3),
        return_exceptions=True,
    )
    for r in results:
        if isinstance(r, Exception):
            print(f"Error: {r}")    # Error: Failed on 2
        else:
            print(f"OK: {r}")       # OK: 10, OK: 30

In [22]:
await handle_errors()

OK: 10
Error: Failed on 2
OK: 30
